# B4 jeepney RL baseline

Traffic-biased quadrilateral baselines on the physical drivable street network.
The minimum-area rule is inspired by polygonal morphology bounds discussed in
https://arxiv.org/html/2603.28385v1.


## 1. Traffic-Biased Quadrilateral Baseline Generator

Builds traffic-weighted quadrilateral routes on the physical drive graph, then stitches the shortest physical path between the four selected anchors.

In [7]:
import pandas as pd
import secrets
from pathlib import Path
from types import SimpleNamespace

from IPython.display import IFrame, display
from tqdm.auto import tqdm

from _bootstrap import ROOT
from utils.route_generation import BaselineRoute, BaselineRouteGenerator, JeepneyRouteEnv, SystemicFitnessEvaluator, calculate_route_fitness, train_route_agent
from utils.travel_graph import JeepneySystem

OUTPUT_DIR = Path(r"C:\Users\lifei\OneDrive\Desktop\Thesis\Thesis Repository\Thesis") / "results" / "B4_baseline_routes"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NUM_ROUTES = 20
random_seed = secrets.randbits(32)
generator = BaselineRouteGenerator(
    min_area_m2=2_000_000,
    anchor_pool_size=96,
    max_attempts=500,
    seed=random_seed,
)

routes = generator.generate_routes(NUM_ROUTES, route_prefix="B4", seed=random_seed)
route_system = JeepneySystem.from_routes(routes)

summary = pd.DataFrame(
    [
        {
            "route_id": route.route_id,
            "anchors": list(route.ordered_anchor_node_ids),
            "area_m2": round(route.polygon_area_m2, 0),
            "length_m": round(route.path_length_m, 0),
            "attempt": route.attempt_index,
        }
        for route in routes
    ]
)
summary


KeyboardInterrupt: 

## 2. RL Environment and Coordinate-Invariant Geometric Embeddings

Uses only the primal drivable network to build route geometry step by step, while encoding turn structure, sinuosity, and origin-relative features without absolute node IDs.

In [ ]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)
env = JeepneyRouteEnv(
    drive_graph_raw=generator.drive_graph_raw,
    drive_graph_proj=generator.drive_graph_proj,
    seed=random_seed,
    max_steps=12,
)

def print_observation(obs: dict[str, np.ndarray]) -> None:
    print('shape:', obs['shape'])
    print('history:', obs['history'])
    print('topology:', obs['topology'])
    print('demand:', obs['demand'])
    print('global:', obs['global'])
    print('candidates:')
    for candidate_index, row in enumerate(obs['candidates']):
        print(f'  {candidate_index}: {row}')
    print('mask:', obs['action_mask'])

obs, info = env.reset()
print('reset state vector length:', info['state_vector'].shape[0])
print_observation(obs)

rng = np.random.default_rng(random_seed)
for step_index in range(5):
    valid_actions = np.flatnonzero(obs['action_mask'][:-1])
    action = int(rng.choice(valid_actions)) if len(valid_actions) else env.max_candidates
    obs, reward, terminated, truncated, info = env.step(action)
    print(f'step {step_index + 1}: action={action}, reward={reward:.3f}, terminated={terminated}, truncated={truncated}')
    print('turn angle rad:', info['turn_angle_rad'])
    print('sinuosity index:', info['sinuosity_index'])
    print('distance to origin m:', info['distance_to_origin_m'])
    print('bearing to origin rad:', info['bearing_to_origin_rad'])
    print('route area m2:', info['route_area_m2'])
    print('state vector:', info['state_vector'])
    print_observation(obs)
    if terminated or truncated:
        break


reset state vector length: 91
shape: [0. 0. 1. 0. 0. 0. 0.]
history: [0. 0. 0. 0. 0. 0. 0. 0.]
topology: [0.5   0.5   0.    0.5   0.556]
demand: [0.226 0.357 0.511 0.285]
global: [0.    0.    1.    0.    0.    0.    0.    0.    0.    0.    0.    0.
 0.    0.    0.    0.5   0.5   0.    0.5   0.556 0.226 0.357 0.511 0.285]
candidates:
  0: [-0.389 -0.921  0.011  0.667  0.     0.511  0.285  0.     1.     0.   ]
  1: [-0.295  0.956  0.017  0.5    0.     0.28   0.053  0.     0.     0.   ]
  2: [0.618 0.786 0.009 0.5   0.    0.28  0.053 0.    0.    0.   ]
  3: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  4: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  5: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
mask: [1. 1. 1. 0. 0. 0. 0.]
step 1: action=1, reward=0.250, terminated=False, truncated=False
turn angle rad: 0.0
sinuosity index: 1.0
distance to origin m: 117.46934991757044
bearing to origin rad: 3.141592653589793
route area m2: 0.0
state vector: [ 0.002  0.    -1.     0.002  0.002  0.     0.     0.     0.     0.
  0.     1.   

: 

: 

In [ ]:
route_edges = pd.DataFrame(
    [(int(u), int(v), "start_walk") for u, v in generator.drive_graph_raw.edges()],
    columns=["u", "v", "edge_type"],
)
route_manager = SimpleNamespace(
    edges=route_edges,
    _node_coords={
        int(row.base_node_id): (float(row.lat), float(row.lon))
        for row in generator.node_table.itertuples(index=False)
    },
)

def build_route_encoding(route_id: str) -> str:
    return f"route_id: {route_id}"

def interpret_embeddings(route: BaselineRoute) -> str:
    """Generate human-readable interpretations of route geometry."""
    route_path = route.path_node_ids
    if not route_path or len(route_path) < 2:
        return "Route too short to analyze."
    
    lines = []
    coords = route.path_latlon
    
    area = route.polygon_area_m2
    length = route.path_length_m
    area_to_length = area / max(length, 1.0)
    
    if area_to_length > 1000:
        lines.append("shape: Compact, efficient use of space")
    elif area_to_length > 200:
        lines.append("shape: Moderate coverage relative to distance")
    else:
        lines.append("shape: Linear route with extended length")
    
    node_count = len(route_path)
    if node_count > 50:
        lines.append("topology: High node density, complex route structure")
    elif node_count > 25:
        lines.append("topology: Moderate complexity with multiple waypoints")
    else:
        lines.append("topology: Simple, direct route path")
    
    if area > 50_000_000:
        lines.append("demand: Large service area, potentially high demand coverage")
    elif area > 10_000_000:
        lines.append("demand: Medium service area with moderate demand potential")
    else:
        lines.append("demand: Compact service area with focused demand")
    
    if len(coords) >= 2:
        import math
        start = coords[0]
        end = coords[-1]
        straight_dist = math.sqrt((end[0] - start[0])**2 + (end[1] - start[1])**2) * 111000
        actual_path = length
        sinuosity = actual_path / max(straight_dist, 1.0)
        
        if sinuosity > 1.5:
            lines.append("history: Winding route with many turns")
        elif sinuosity > 1.1:
            lines.append("history: Moderate turning pattern")
        else:
            lines.append("history: Direct path with minimal deviations")
    
    return chr(10).join(lines) if lines else "No embedding data available."

def format_route_fitness(result) -> str:
    return (
        f"standalone fitness: reward={result.reward:.3f} | avg_gtc={result.average_gtc:.3f} | "
        f"passenger_std={result.passenger_gtc_std:.3f}"
    )

def format_systemic_fitness(result) -> str:
    return (
        f"systemic fitness: avg_gtc={result.average_gtc:.3f} ± {result.std_gtc:.3f} | "
        f"avg_passenger_std={result.average_passenger_gtc_std:.3f} ± {result.std_passenger_gtc_std:.3f}"
    )

NOTE_SYSTEMIC_EVAL_TEST_MEAN = 3  # Lightweight note-time sample; increase later for final analysis.
NOTE_SYSTEMIC_EVAL_TEST_STD = 1
NOTE_SYSTEMIC_BACKGROUND_ROUTE_MEAN = 1
NOTE_SYSTEMIC_BACKGROUND_ROUTE_STD = 0
FULL_SYSTEMIC_EVAL_TEST_MEAN = 10  # Temporary placeholder; increase later for final sensitivity analysis.
FULL_SYSTEMIC_EVAL_TEST_STD = 5
FULL_SYSTEMIC_BACKGROUND_ROUTE_MEAN = 2
FULL_SYSTEMIC_BACKGROUND_ROUTE_STD = 1

note_systemic_evaluator = SystemicFitnessEvaluator(
    passenger_map=generator.passenger_map,
    drive_graph_raw=generator.drive_graph_raw,
    drive_graph_proj=generator.drive_graph_proj,
    evaluation_test_mean=NOTE_SYSTEMIC_EVAL_TEST_MEAN,
    evaluation_test_std=NOTE_SYSTEMIC_EVAL_TEST_STD,
    background_route_mean=NOTE_SYSTEMIC_BACKGROUND_ROUTE_MEAN,
    background_route_std=NOTE_SYSTEMIC_BACKGROUND_ROUTE_STD,
    seed=random_seed,
)
full_systemic_evaluator = SystemicFitnessEvaluator(
    passenger_map=generator.passenger_map,
    drive_graph_raw=generator.drive_graph_raw,
    drive_graph_proj=generator.drive_graph_proj,
    evaluation_test_mean=FULL_SYSTEMIC_EVAL_TEST_MEAN,
    evaluation_test_std=FULL_SYSTEMIC_EVAL_TEST_STD,
    background_route_mean=FULL_SYSTEMIC_BACKGROUND_ROUTE_MEAN,
    background_route_std=FULL_SYSTEMIC_BACKGROUND_ROUTE_STD,
    seed=random_seed,
)

standalone_fitness_cache: dict[tuple[int, ...], object] = {}
systemic_fitness_cache: dict[tuple[int, ...], object] = {}

def route_signature(route) -> tuple[int, ...]:
    return tuple(int(node_id) for node_id in route.path_node_ids)

def get_standalone_fitness(route):
    key = route_signature(route)
    cached = standalone_fitness_cache.get(key)
    if cached is None:
        cached = calculate_route_fitness(
            route.path_node_ids,
            passenger_map=generator.passenger_map,
            drive_graph_raw=generator.drive_graph_raw,
            drive_graph_proj=generator.drive_graph_proj,
            seed=random_seed,
            batch_size=5,
        )
        standalone_fitness_cache[key] = cached
    return cached

def get_systemic_fitness(route, *, use_full: bool = False):
    key = route_signature(route)
    cache = systemic_fitness_cache
    cached = cache.get((key, use_full))
    if cached is None:
        evaluator = full_systemic_evaluator if use_full else note_systemic_evaluator
        cached = evaluator.evaluate(route, seed=random_seed)
        cache[(key, use_full)] = cached
    return cached

def build_route_nodes(summary_df: pd.DataFrame, routes: list) -> dict[str, dict[str, str]]:
    notes: dict[str, dict[str, str]] = {}
    route_by_id = {r.route_id: r for r in routes}
    total = len(summary_df)
    if total == 0:
        return notes
    
    for row in tqdm(summary_df.itertuples(index=False), total=total, desc="Building route notes"):
        route = route_by_id.get(row.route_id)
        interpretation = interpret_embeddings(route) if route else "Route data unavailable."
        if route is not None:
            standalone = get_standalone_fitness(route)
            systemic = get_systemic_fitness(route)
            interpretation = (
                f"{format_route_fitness(standalone)}\n"
                f"{format_systemic_fitness(systemic)}\n\n"
                f"{interpretation}"
            )
        notes[row.route_id] = {
            "encoding": build_route_encoding(row.route_id),
            "interpretation": interpretation,
        }
    return notes

route_notes = build_route_nodes(summary, routes)


Building route notes:   0%|          | 0/20 [00:00<?, ?it/s]

: 

: 

## 3. 3-Layer Graph Reward Formulation
Connect the generated physical shape to the 3-layer behavioral passenger graph to calculate the fitness score (Reward).

In [ ]:
comparison_generator = BaselineRouteGenerator(
    min_area_m2=2_000_000,
    anchor_pool_size=96,
    max_attempts=500,
    seed=1,
)
comparison_routes = comparison_generator.generate_routes(2, route_prefix='CMP', seed=1)
good_shape = comparison_routes[0].path_node_ids
bad_shape = comparison_routes[1].path_node_ids

good_result = calculate_route_fitness(
    good_shape,
    passenger_map=comparison_generator.passenger_map,
    drive_graph_raw=comparison_generator.drive_graph_raw,
    drive_graph_proj=comparison_generator.drive_graph_proj,
    seed=1,
    batch_size=5,
)
bad_result = calculate_route_fitness(
    bad_shape,
    passenger_map=comparison_generator.passenger_map,
    drive_graph_raw=comparison_generator.drive_graph_raw,
    drive_graph_proj=comparison_generator.drive_graph_proj,
    seed=1,
    batch_size=5,
)

comparison = pd.DataFrame(
    [
        {"route_shape": "good", "reward": round(good_result.reward, 3), "avg_gtc": round(good_result.average_gtc, 3), "unserved": good_result.unserved_passenger_count},
        {"route_shape": "bad", "reward": round(bad_result.reward, 3), "avg_gtc": round(bad_result.average_gtc, 3), "unserved": bad_result.unserved_passenger_count},
    ]
)
print(f"Good route reward: {good_result.reward:.3f}")
print(f"Bad route reward: {bad_result.reward:.3f}")
comparison


Good route reward: -174.980
Bad route reward: -176.625


,route_shape,reward,avg_gtc,unserved
0,good,-174.980,174.980,0
1,bad,-176.625,176.625,0


: 

: 

## 4. Systemic Synergy Evaluation Utility
Build a statistical evaluation utility that tests a candidate route across multiple background networks to measure true system synergy.

In [ ]:
systemic_result = full_systemic_evaluator.evaluate(routes[0], seed=random_seed)
print(f"Systemic average GTC: {systemic_result.average_gtc:.3f}")
print(f"Systemic GTC std dev: {systemic_result.std_gtc:.3f}")
print(f"Passenger GTC std avg: {systemic_result.average_passenger_gtc_std:.3f}")
print(f"Tests used: {systemic_result.n_tests}")


Systemic average GTC: 6152.394
Systemic GTC std dev: 1312.061
Passenger GTC std avg: 14689.644
Tests used: 14


: 

: 

## For Evaluation: Export HTML Tester

In [ ]:
html_path = route_system.export_route_toggle_html(
    route_manager,
    OUTPUT_DIR / "B4_route_explorer.html",
    title=f"B4 Jeepney Routes ({NUM_ROUTES})",
    route_notes=route_notes,
)
print(html_path)
display(IFrame(html_path.as_uri(), width=1200, height=900))


C:\Users\lifei\OneDrive\Desktop\Thesis\Thesis Repository\Thesis\results\B4_baseline_routes\B4_route_explorer.html


: 

: 

## 5. Standalone Rejection Sampling Setup
This cell rebuilds the route generator and systemic evaluator so part 2 can run without any earlier notebook state.


In [ ]:
from pathlib import Path
import sys

import csv
import importlib
import numpy as np
import pandas as pd
import yaml
from IPython.display import display
from tqdm.auto import tqdm

REPO_ROOT = Path(r"C:\Users\lifei\OneDrive\Desktop\Thesis\Thesis Repository\Thesis")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import utils.route_generation.baseline_route_generator as baseline_route_generator_module
import utils.route_generation.systemic_fitness_evaluator as systemic_fitness_evaluator_module
importlib.reload(baseline_route_generator_module)
importlib.reload(systemic_fitness_evaluator_module)
from utils.route_generation.baseline_route_generator import BaselineRouteGenerator
from utils.route_generation.systemic_fitness_evaluator import SystemicFitnessEvaluator

SAMPLE_SIZE = 5000
GOOD_PERCENTILE = 95.0
ELITE_PERCENTILE = 99.0
RISK_AVERSION_K = 1.0
ROBUSTNESS_CV_CEILING = 0.20
SCREENING_TESTS = 1
VALIDATION_TESTS = 10
SCREENING_BACKGROUND_ROUTE_MEAN = 1
SCREENING_BACKGROUND_ROUTE_STD = 0
VALIDATION_BACKGROUND_ROUTE_MEAN = 2
VALIDATION_BACKGROUND_ROUTE_STD = 1
RANDOM_SEED = 20260422
OUTPUT_YAML = REPO_ROOT / "configs" / "B4_rejection_sampling.yaml"

sampler = BaselineRouteGenerator(
    min_area_m2=2_000_000,
    anchor_pool_size=96,
    max_attempts=500,
    seed=RANDOM_SEED,
)
screening_evaluator = SystemicFitnessEvaluator(
    passenger_map=sampler.passenger_map,
    drive_graph_raw=sampler.drive_graph_raw,
    drive_graph_proj=sampler.drive_graph_proj,
    evaluation_test_mean=SCREENING_TESTS,
    evaluation_test_std=0.0,
    background_route_mean=SCREENING_BACKGROUND_ROUTE_MEAN,
    background_route_std=SCREENING_BACKGROUND_ROUTE_STD,
    seed=RANDOM_SEED,
)
validation_evaluator = SystemicFitnessEvaluator(
    passenger_map=sampler.passenger_map,
    drive_graph_raw=sampler.drive_graph_raw,
    drive_graph_proj=sampler.drive_graph_proj,
    evaluation_test_mean=VALIDATION_TESTS,
    evaluation_test_std=0.0,
    background_route_mean=VALIDATION_BACKGROUND_ROUTE_MEAN,
    background_route_std=VALIDATION_BACKGROUND_ROUTE_STD,
    seed=RANDOM_SEED,
)
route_seed_rng = np.random.default_rng(RANDOM_SEED)
validation_seed_rng = np.random.default_rng(RANDOM_SEED + 1)

def risk_adjusted_score(result) -> float:
    return float(-(float(result.average_gtc) + (RISK_AVERSION_K * float(result.std_gtc))))

def coefficient_of_variation(result) -> float:
    mean_gtc = float(result.average_gtc)
    if mean_gtc <= 0:
        return float("inf")
    return float(result.std_gtc / mean_gtc)

print(f"Standalone setup ready: {SAMPLE_SIZE:,} candidates, P{GOOD_PERCENTILE:.0f}/P{ELITE_PERCENTILE:.0f} thresholds")


Standalone setup ready: 5,000 candidates, P95/P99 thresholds


: 

: 

### 5.1 Sample and screen candidate routes
This cell generates 10,000 candidates with a progress bar and scores each one against the systemic evaluator.


In [ ]:
screening_rows = []
candidate_by_id = {}

for index in tqdm(range(SAMPLE_SIZE), desc="Sampling candidate routes", unit="route"):
    while True:
        route_seed = int(route_seed_rng.integers(0, np.iinfo(np.int32).max))
        try:
            route = sampler.generate_route(route_id=f"B4Q{index + 1:05d}", seed=route_seed)
            break
        except ValueError:
            continue

    candidate_by_id[route.route_id] = route
    result = screening_evaluator.evaluate(route, n_tests=SCREENING_TESTS, seed=route_seed)
    screening_rows.append(
        {
            "route_id": route.route_id,
            "average_gtc": float(result.average_gtc),
            "std_gtc": float(result.std_gtc),
            "risk_adjusted_score": risk_adjusted_score(result),
            "reward": float(result.reward),
            "average_passenger_gtc_std": float(result.average_passenger_gtc_std),
            "n_tests": int(result.n_tests),
            "area_m2": float(route.polygon_area_m2),
            "length_m": float(route.path_length_m),
            "attempt": int(route.attempt_index),
            "coefficient_of_variation": coefficient_of_variation(result),
        }
    )

screening_scores = pd.DataFrame(screening_rows).sort_values("risk_adjusted_score", ascending=False).reset_index(drop=True)
display(screening_scores.head(10))


Sampling candidate routes:   0%|          | 0/5000 [00:00<?, ?route/s]

KeyboardInterrupt: 

: 

: 

### 5.2 Filter for robust routes
This cell applies percentile cutoffs, the mean-plus-two-sigma check, and the coefficient-of-variation ceiling.


In [ ]:
score_values = screening_scores["risk_adjusted_score"].to_numpy(dtype=float)
mean_gtc = float(screening_scores["average_gtc"].mean())
std_gtc = float(screening_scores["std_gtc"].mean())
mean_risk_adjusted_score = float(score_values.mean())
std_risk_adjusted_score = float(score_values.std(ddof=0))
good_threshold = float(np.percentile(score_values, GOOD_PERCENTILE))
elite_threshold = float(np.percentile(score_values, ELITE_PERCENTILE))
sigma_threshold = float(mean_risk_adjusted_score + 2.0 * std_risk_adjusted_score)
robustness_limit = float(ROBUSTNESS_CV_CEILING)

good_scores = screening_scores.loc[(screening_scores["risk_adjusted_score"] >= good_threshold) & (screening_scores["coefficient_of_variation"] <= robustness_limit)].copy().reset_index(drop=True)
elite_scores = screening_scores.loc[(screening_scores["risk_adjusted_score"] >= elite_threshold) & (screening_scores["coefficient_of_variation"] <= robustness_limit)].copy().reset_index(drop=True)
sigma_scores = screening_scores.loc[(screening_scores["risk_adjusted_score"] >= sigma_threshold) & (screening_scores["coefficient_of_variation"] <= robustness_limit)].copy().reset_index(drop=True)
good_routes = [candidate_by_id[row.route_id] for row in good_scores.itertuples(index=False)]

print(f"Accepted {len(good_routes)} routes at P{GOOD_PERCENTILE:.0f}")
print(f"Risk-adjusted threshold: {good_threshold:.3f}")
print(f"Mean + 2σ threshold: {sigma_threshold:.3f}")
print(f"Robustness ceiling (CV): {robustness_limit:.2f}")
display(good_scores.head(10))


Accepted 249 routes at P95
Risk-adjusted threshold: -3720.032
Mean + 2σ threshold: -3104.456
Robustness ceiling (CV): 0.20


,route_id,average_gtc,std_gtc,risk_adjusted_score,reward,average_passenger_gtc_std,n_tests,area_m2,length_m,attempt,coefficient_of_variation
0,B4Q01373,584.016285,0.0,-584.016285,-584.016285,1856.534064,1,4.165664e+06,12952.829455,3,0.0
1,B4Q00947,1437.144167,0.0,-1437.144167,-1437.144167,5647.905898,1,3.320708e+07,38808.547887,1,0.0
2,B4Q03418,1456.575984,0.0,-1456.575984,-1456.575984,4733.531899,1,1.232111e+07,41967.340706,1,0.0
3,B4Q03413,1613.325502,0.0,-1613.325502,-1613.325502,6390.723852,1,6.034260e+07,67157.961309,1,0.0
4,B4Q01339,1788.059585,0.0,-1788.059585,-1788.059585,6744.595320,1,5.933165e+07,66737.776649,1,0.0
5,B4Q04581,1992.667177,0.0,-1992.667177,-1992.667177,7685.344781,1,1.173095e+08,69086.150437,1,0.0
6,B4Q00276,2139.862964,0.0,-2139.862964,-2139.862964,6701.586275,1,4.148586e+07,67972.672295,1,0.0
7,B4Q00097,2189.740142,0.0,-2189.740142,-2189.740142,7765.408860,1,1.468155e+08,70868.537385,1,0.0
8,B4Q02904,2255.886026,0.0,-2255.886026,-2255.886026,7389.822856,1,7.076008e+06,35271.803179,1,0.0
9,B4Q00479,2264.065078,0.0,-2264.065078,-2264.065078,8418.232429,1,1.378964e+07,70194.717999,1,0.0


: 

: 

### 5.3 Validate and export
This cell re-evaluates the accepted routes with a broader systemic pass, then saves the run settings to YAML.


In [ ]:
validation_rows = []
for route in tqdm(good_routes, desc="Validating accepted routes", unit="route"):
    validation_seed = int(validation_seed_rng.integers(0, np.iinfo(np.int32).max))
    validation = validation_evaluator.evaluate(route, n_tests=VALIDATION_TESTS, seed=validation_seed)
    validation_rows.append(
        {
            "route_id": route.route_id,
            "average_gtc": float(validation.average_gtc),
            "std_gtc": float(validation.std_gtc),
            "risk_adjusted_score": risk_adjusted_score(validation),
            "reward": float(validation.reward),
            "average_passenger_gtc_std": float(validation.average_passenger_gtc_std),
            "n_tests": int(validation.n_tests),
            "coefficient_of_variation": coefficient_of_variation(validation),
        }
    )

validation_scores = pd.DataFrame(
    validation_rows,
    columns=[
        "route_id",
        "average_gtc",
        "std_gtc",
        "risk_adjusted_score",
        "reward",
        "average_passenger_gtc_std",
        "n_tests",
        "coefficient_of_variation",
    ],
).sort_values("risk_adjusted_score", ascending=False).reset_index(drop=True)

yaml_payload = {
    "experiment": "B4_rejection_sampling",
    "higher_is_better": True,
    "sample_size": SAMPLE_SIZE,
    "random_seed": RANDOM_SEED,
    "risk_aversion_k": RISK_AVERSION_K,
    "robustness_cv_ceiling": ROBUSTNESS_CV_CEILING,
    "route_generator": {
        "min_area_m2": 2_000_000,
        "anchor_pool_size": 96,
        "max_attempts": 500,
        "route_prefix": "B4Q",
    },
    "screening": {
        "good_percentile": GOOD_PERCENTILE,
        "elite_percentile": ELITE_PERCENTILE,
        "tests": SCREENING_TESTS,
        "background_route_mean": SCREENING_BACKGROUND_ROUTE_MEAN,
        "background_route_std": SCREENING_BACKGROUND_ROUTE_STD,
        "percentile_threshold": good_threshold,
        "elite_threshold": elite_threshold,
        "mean_plus_2sigma_threshold": sigma_threshold,
        "robustness_ceiling": robustness_limit,
    },
    "validation": {
        "tests": VALIDATION_TESTS,
        "background_route_mean": VALIDATION_BACKGROUND_ROUTE_MEAN,
        "background_route_std": VALIDATION_BACKGROUND_ROUTE_STD,
    },
    "summary": {
        "mean_gtc": mean_gtc,
        "std_gtc": std_gtc,
        "mean_risk_adjusted_score": mean_risk_adjusted_score,
        "std_risk_adjusted_score": std_risk_adjusted_score,
        "good_route_count": int(len(good_scores)),
        "elite_route_count": int(len(elite_scores)),
        "sigma_route_count": int(len(sigma_scores)),
        "best_route_id": str(screening_scores.iloc[0]["route_id"]),
        "best_risk_adjusted_score": float(screening_scores.iloc[0]["risk_adjusted_score"]),
        "best_average_gtc": float(screening_scores.iloc[0]["average_gtc"]),
        "best_std_gtc": float(screening_scores.iloc[0]["std_gtc"]),
        "validation_mean_gtc": float(validation_scores["average_gtc"].mean()) if not validation_scores.empty else None,
        "validation_std_gtc": float(validation_scores["average_gtc"].std(ddof=0)) if len(validation_scores) > 1 else 0.0 if len(validation_scores) == 1 else None,
        "validation_mean_risk_adjusted_score": float(validation_scores["risk_adjusted_score"].mean()) if not validation_scores.empty else None,
        "validation_std_risk_adjusted_score": float(validation_scores["risk_adjusted_score"].std(ddof=0)) if len(validation_scores) > 1 else 0.0 if len(validation_scores) == 1 else None,
        "validation_best_route_id": str(validation_scores.iloc[0]["route_id"]) if not validation_scores.empty else None,
    },
}
OUTPUT_YAML.write_text(yaml.safe_dump(yaml_payload, sort_keys=False, allow_unicode=False), encoding="utf-8")

print(f"Validation routes: {len(validation_scores)}")
print(f"YAML saved to: {OUTPUT_YAML}")
display(validation_scores.head(10))


Validating accepted routes:   0%|          | 0/249 [00:00<?, ?route/s]

Validation routes: 249
YAML saved to: C:\Users\lifei\OneDrive\Desktop\Thesis\Thesis Repository\Thesis\configs\B4_rejection_sampling.yaml


,route_id,average_gtc,std_gtc,risk_adjusted_score,reward,average_passenger_gtc_std,n_tests,coefficient_of_variation
0,B4Q00856,5145.015613,595.755897,-5740.771511,-5740.771511,12829.769288,10,0.115793
1,B4Q01378,5093.571467,799.238544,-5892.810011,-5892.810011,13118.263282,10,0.156911
2,B4Q02357,5291.279273,723.088061,-6014.367334,-6014.367334,13780.367056,10,0.136657
3,B4Q02260,5064.054653,958.749562,-6022.804214,-6022.804214,13461.513261,10,0.189324
4,B4Q04971,5238.917559,785.871973,-6024.789532,-6024.789532,13298.617924,10,0.150007
5,B4Q04052,5226.821644,802.088618,-6028.910262,-6028.910262,13274.567208,10,0.153456
6,B4Q00790,5232.022652,826.592274,-6058.614926,-6058.614926,13406.269579,10,0.157987
7,B4Q00996,4956.322340,1105.929948,-6062.252287,-6062.252287,13261.620795,10,0.223135
8,B4Q03400,5417.665172,661.830636,-6079.495808,-6079.495808,13627.323008,10,0.122162
9,B4Q03563,4970.215794,1149.257633,-6119.473426,-6119.473426,12675.074836,10,0.231229


: 

: 

### Optimization log

- Added Cython helpers for anchor ordering, cost reduction, nearest-node lookup, and shortest-path edge reconstruction.
- Added Cython build support and kept a pure-Python fallback for parity.
- Removed eager graph construction in validation and reused cached node geometry plus per-edge weights in the travel graph.
- Cached passenger nearest-node and shortest-path resolution so path prep does not repeat the same lookup twice.
- Reused the systemic validation process pool across route checks, but kept small batches on the single-process path so multiprocessing overhead does not dominate.
- Kept part 3 append-only and resumable, with progress bars and JSONL-first persistence.


## 6. Batch Good Route Library
This section keeps the old 5000-sample workflow: generate a batch, log every route to CSV immediately, then flag the top 5% by the part 2 score after the batch finishes.


### 6.1 Standalone setup
The next cell reloads the part 2 thresholds from disk, recreates the generator and systemic evaluators, and prepares the batch snapshot path so this part can run on its own.


In [14]:
from pathlib import Path
import csv
import json
import secrets
import sys

import numpy as np
import pandas as pd
import yaml
from IPython.display import display
from tqdm.auto import tqdm

REPO_ROOT = Path(r"C:\Users\lifei\OneDrive\Desktop\Thesis\Thesis Repository\Thesis")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils.route_generation import BaselineRouteGenerator, SystemicFitnessEvaluator

# ===================== CONFIGURATION =====================
SAMPLE_SIZE = 10000
LIBRARY_ROUTE_PREFIX = "ROUTE"
GOOD_PERCENTILE = 95.0
ELITE_PERCENTILE = 99.0
RISK_AVERSION_K = 1.0
ROBUSTNESS_CV_CEILING = 0.20
RUN_SEED = 1670006087
SCREENING_TESTS = 1
SCREENING_BACKGROUND_ROUTE_MEAN = 1
SCREENING_BACKGROUND_ROUTE_STD = 0
LIBRARY_REFRESH_EVERY = 25
SCREENING_SCORE_THRESHOLD = -3720.032388103914  # P95 threshold from part 2

# Paths
OUTPUT_DIR = REPO_ROOT / "results" / "B4_good_routes"
OUTPUT_CSV = OUTPUT_DIR / "good_routes.csv"
OUTPUT_YAML = REPO_ROOT / "configs" / "B4_good_route_library.yaml"
PART2_CONFIG = REPO_ROOT / "configs" / "B4_rejection_sampling.yaml"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ===================== HELPER FUNCTIONS =====================
def risk_adjusted_score(result) -> float:
    """Calculate risk-adjusted score: -(mean_gtc + k * std_gtc)"""
    return float(-(float(result.average_gtc) + (RISK_AVERSION_K * float(result.std_gtc))))

def coefficient_of_variation(result) -> float:
    """Calculate coefficient of variation: std / mean"""
    mean_gtc = float(result.average_gtc)
    if mean_gtc <= 0:
        return float("inf")
    return float(result.std_gtc / mean_gtc)

# ===================== CREATE GENERATOR & EVALUATOR =====================
sampler = BaselineRouteGenerator(
    min_area_m2=2_000_000,
    anchor_pool_size=96,
    max_attempts=500,
    seed=RUN_SEED,
)

screening_evaluator = SystemicFitnessEvaluator(
    passenger_map=sampler.passenger_map,
    drive_graph_raw=sampler.drive_graph_raw,
    drive_graph_proj=sampler.drive_graph_proj,
    evaluation_test_mean=SCREENING_TESTS,
    evaluation_test_std=0.0,
    background_route_mean=SCREENING_BACKGROUND_ROUTE_MEAN,
    background_route_std=SCREENING_BACKGROUND_ROUTE_STD,
    seed=RUN_SEED,
)

print(f"  Setup complete: sampler and screening_evaluator ready")
print(f"  Output: {OUTPUT_CSV}")
print(f"  Screening threshold (P{GOOD_PERCENTILE:.0f}): {SCREENING_SCORE_THRESHOLD:.3f}")

  Setup complete: sampler and screening_evaluator ready
  Output: C:\Users\lifei\OneDrive\Desktop\Thesis\Thesis Repository\Thesis\results\B4_good_routes\good_routes.csv
  Screening threshold (P95): -3720.032


In [15]:
# Part 3: Initialize library state and configuration
from pathlib import Path
import yaml
import json
import csv
import numpy as np
import pandas as pd
import os

# ===================== CONFIGURATION =====================
SAMPLE_SIZE = 10000
GOOD_PERCENTILE = 95.0
RUN_SEED = 1670006087
RISK_AVERSION_K = 1.0
ROBUSTNESS_CV_CEILING = 0.20
SCREENING_SCORE_THRESHOLD = -3720.032388103914
SCREENING_TESTS = 1
LIBRARY_REFRESH_EVERY = 25

OUTPUT_DIR = REPO_ROOT / "results" / "B4_good_routes"
OUTPUT_CSV = OUTPUT_DIR / "good_routes.csv"
OUTPUT_YAML = REPO_ROOT / "configs" / "B4_good_route_library.yaml"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ===================== HELPER FUNCTIONS =====================
def route_key(route):
    return tuple(int(node_id) for node_id in route.path_node_ids)

def load_existing_library(csv_path):
    if csv_path.exists():
        try:
            df = pd.read_csv(csv_path)
            for col in ['route_node_ids', 'anchor_node_ids', 'ordered_anchor_node_ids', 'path_latlon', 'anchor_latlon']:
                if col in df.columns:
                    df[col] = df[col].apply(lambda x: json.loads(x) if isinstance(x, str) else x)
            return df
        except Exception as e:
            print(f"Warning: Could not load CSV: {e}")
            return pd.DataFrame()
    return pd.DataFrame()

def append_records_to_csv(records):
    """Append records to CSV using global OUTPUT_CSV path"""
    if not records:
        return
    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    rows_to_write = []
    for record in records:
        row = record.copy()
        for json_col in ['route_node_ids', 'anchor_node_ids', 'ordered_anchor_node_ids', 'path_latlon', 'anchor_latlon']:
            if json_col in row and isinstance(row[json_col], (list, dict)):
                row[json_col] = json.dumps(row[json_col], ensure_ascii=False)
        rows_to_write.append(row)
    
    file_exists = OUTPUT_CSV.exists()
    fieldnames = list(rows_to_write[0].keys()) if rows_to_write else []
    mode = 'a' if file_exists else 'w'
    with open(OUTPUT_CSV, mode, newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        writer.writerows(rows_to_write)
        f.flush()
        os.fsync(f.fileno())  # Force disk write

def flush_pending_records(library_records, pending):
    for record in pending:
        for json_col in ['route_node_ids', 'anchor_node_ids', 'ordered_anchor_node_ids', 'path_latlon', 'anchor_latlon']:
            if json_col in record and isinstance(record[json_col], str):
                try:
                    record[json_col] = json.loads(record[json_col])
                except:
                    pass
    library_records.extend(pending)
    return library_records

def live_top_percentile_summary(library_records, percentile=95.0):
    """Generate live summary table that can be displayed and updated"""
    if not library_records or len(library_records) == 0:
        df = pd.DataFrame({
            'routes_logged': [0],
            'p95_threshold': [np.nan],
            'routes_at_or_above_p95': [0],
            'top_p95_mean_score': [np.nan],
            'top_p95_std_score': [np.nan],
            'best_route_id': ['--'],
            'best_score': [np.nan],
        })
        return df
    
    try:
        # Ensure JSON columns are parsed before creating DataFrame
        parsed_records = []
        for record in library_records:
            r = dict(record)
            for json_col in ['route_node_ids', 'anchor_node_ids', 'ordered_anchor_node_ids', 'path_latlon', 'anchor_latlon']:
                if json_col in r and isinstance(r[json_col], str):
                    try:
                        r[json_col] = json.loads(r[json_col])
                    except:
                        pass
            parsed_records.append(r)
        df = pd.DataFrame(parsed_records)
        if 'screening_risk_adjusted_score' not in df.columns or len(df) == 0:
            df_out = pd.DataFrame({
                'routes_logged': [len(library_records)],
                'p95_threshold': [np.nan],
                'routes_at_or_above_p95': [0],
                'top_p95_mean_score': [np.nan],
                'top_p95_std_score': [np.nan],
                'best_route_id': ['--'],
                'best_score': [np.nan],
            })
            return df_out
        
        scores = df['screening_risk_adjusted_score'].to_numpy(dtype=float)
        threshold = float(np.percentile(scores, percentile))
        top_mask = scores >= threshold
        top_routes = df[top_mask]
        
        summary = pd.DataFrame({
            'routes_logged': [len(df)],
            'p95_threshold': [threshold],
            'routes_at_or_above_p95': [int(top_mask.sum())],
            'top_p95_mean_score': [float(top_routes['screening_risk_adjusted_score'].mean()) if len(top_routes) > 0 else np.nan],
            'top_p95_std_score': [float(top_routes['screening_risk_adjusted_score'].std()) if len(top_routes) > 1 else np.nan],
            'best_route_id': [str(df.iloc[0]['route_id']) if len(df) > 0 else '--'],
            'best_score': [df.iloc[0]['screening_risk_adjusted_score'] if len(df) > 0 else np.nan],
        })
        return summary
    except Exception as e:
        print(f"Error in live_top_percentile_summary: {e}")
        return pd.DataFrame({'error': [str(e)]})

def write_library_snapshot(library_df, session_stats):
    if OUTPUT_YAML is None:
        return library_df
    
    manifest = {
        'experiment': 'B4_good_route_library',
        'higher_is_better': True,
        'run_seed': RUN_SEED,
        'risk_aversion_k': RISK_AVERSION_K,
        'robustness_cv_ceiling': ROBUSTNESS_CV_CEILING,
        'summary': {
            'route_count': int(len(library_df)),
            'best_route_id': str(library_df.iloc[0]['route_id']) if len(library_df) > 0 else None,
        },
        'session': session_stats,
    }
    OUTPUT_YAML.write_text(yaml.safe_dump(manifest, sort_keys=False, allow_unicode=False), encoding='utf-8')
    return library_df

# ===================== INITIALIZE STATE =====================
library_records = []
existing_df = load_existing_library(OUTPUT_CSV)
if not existing_df.empty:
    library_records = existing_df.to_dict('records')
    print(f"Loaded {len(library_records)} existing routes")

seen_keys = set()
for record in library_records:
    try:
        nodes = record.get('route_node_ids')
        if isinstance(nodes, str):
            nodes = json.loads(nodes)
        seen_keys.add(tuple(int(n) for n in nodes))
    except:
        pass

SESSION_STATS = {
    'attempts_this_session': 0,
    'screened_this_session': 0,
    'validated_this_session': 0,
    'duplicate_hits_this_session': 0,
    'accepted_this_session': 0,
}
screening_rng = np.random.default_rng(RUN_SEED)

print(f"Setup ready: {len(library_records)} routes loaded, {len(seen_keys)} unique signatures")
print(f"CSV path: {OUTPUT_CSV}")
print(f"YAML path: {OUTPUT_YAML}")

Loaded 39 existing routes
Setup ready: 39 routes loaded, 39 unique signatures
CSV path: C:\Users\lifei\OneDrive\Desktop\Thesis\Thesis Repository\Thesis\results\B4_good_routes\good_routes.csv
YAML path: C:\Users\lifei\OneDrive\Desktop\Thesis\Thesis Repository\Thesis\configs\B4_good_route_library.yaml


### 6.2 Batch generate, log, and inspect
This cell generates 5000 routes, appends every evaluated route to CSV immediately, and updates a live summary of the current top 95 percentile stats.


In [18]:
SAMPLE_SIZE = 10000 - len(library_records)
LIBRARY_ROUTE_PREFIX = "ROUTE"
session_stats = SESSION_STATS
progress_bar = tqdm(total=SAMPLE_SIZE, desc='Generating candidate routes', unit='route', dynamic_ncols=True)

def update_live_stats(library_records):
    summary = live_top_percentile_summary(library_records)
    if summary.empty:
        progress_bar.set_postfix_str('top_p95_mean=nan | top_p95_std=nan')
        return
    row = summary.iloc[0]
    progress_bar.set_postfix(
        top_p95_mean=f"{row['top_p95_mean_score']:.3f}",
        top_p95_std=f"{row['top_p95_std_score']:.3f}",
    )

update_live_stats(library_records)
live_refresh_every = max(int(LIBRARY_REFRESH_EVERY), 1)
batch_flush_every = live_refresh_every
pending_items: list[dict] = []

evaluate_screen = screening_evaluator.evaluate
risk_score = risk_adjusted_score
cov = coefficient_of_variation
screen_threshold = SCREENING_SCORE_THRESHOLD
robustness_limit = ROBUSTNESS_CV_CEILING

try:
    for candidate_index in range(1, SAMPLE_SIZE + 1):
        route_seed = int(screening_rng.integers(0, np.iinfo(np.int32).max))
        while True:
            try:
                route = sampler.generate_route(route_id=f'{LIBRARY_ROUTE_PREFIX}{candidate_index:06d}', seed=route_seed)
                break
            except ValueError:
                route_seed = int(screening_rng.integers(0, np.iinfo(np.int32).max))

        key = route_key(route)
        session_stats['attempts_this_session'] += 1
        is_duplicate = key in seen_keys
        if is_duplicate:
            session_stats['duplicate_hits_this_session'] += 1
        else:
            seen_keys.add(key)

        screening_result = evaluate_screen(route, n_tests=SCREENING_TESTS, seed=route_seed)
        session_stats['screened_this_session'] += 1
        screening_score = risk_score(screening_result)
        screening_cv = cov(screening_result)

        screening_passes = bool(screening_score >= screen_threshold and screening_cv <= robustness_limit)
        record = {
            'route_key': key,
            'route_id': route.route_id,
            'run_seed': RUN_SEED,
            'route_seed': route_seed,
            'is_duplicate': bool(is_duplicate),
            'screening_passes': screening_passes,
            'selected_top95': False,
            'route_node_ids': [int(node_id) for node_id in route.path_node_ids],
            'anchor_node_ids': [int(node_id) for node_id in route.anchor_node_ids],
            'ordered_anchor_node_ids': [int(node_id) for node_id in route.ordered_anchor_node_ids],
            'path_latlon': [[float(lat), float(lon)] for lat, lon in route.path_latlon],
            'anchor_latlon': [[float(lat), float(lon)] for lat, lon in route.anchor_latlon],
            'polygon_area_m2': float(route.polygon_area_m2),
            'path_length_m': float(route.path_length_m),
            'attempt_index': int(route.attempt_index),
            'screening_average_gtc': float(screening_result.average_gtc),
            'screening_std_gtc': float(screening_result.std_gtc),
            'screening_risk_adjusted_score': float(screening_score),
            'screening_reward': float(screening_result.reward),
            'screening_average_passenger_gtc_std': float(screening_result.average_passenger_gtc_std),
            'screening_n_tests': int(screening_result.n_tests),
            'screening_coefficient_of_variation': float(screening_cv),
        }

        pending_items.append(record)

        if len(pending_items) >= batch_flush_every:
            # No validation—append screening results directly to CSV
            append_records_to_csv(pending_items)
            library_records = flush_pending_records(library_records, pending_items)
            update_live_stats(library_records)
            pending_items.clear()

        progress_bar.update(1)

except KeyboardInterrupt:
    print('Stopped by user; rerun this cell to resume from the CSV log.')
finally:
    progress_bar.close()
    if pending_items:
        # Flush remaining pending routes without validation
        append_records_to_csv(pending_items)
        library_records = flush_pending_records(library_records, pending_items)
    library_df = pd.DataFrame.from_records(library_records)
    if not library_df.empty and 'screening_risk_adjusted_score' in library_df.columns:
        final_threshold = float(np.percentile(library_df['screening_risk_adjusted_score'].to_numpy(dtype=float), GOOD_PERCENTILE))
        library_df['selected_top95'] = library_df['screening_risk_adjusted_score'] >= final_threshold
        session_stats['accepted_this_session'] = int(library_df['selected_top95'].sum())
    else:
        final_threshold = None
        session_stats['accepted_this_session'] = 0
        library_df['selected_top95'] = pd.Series(dtype=bool)
    library_df = write_library_snapshot(library_df, session_stats)
    print(f'Routes in library: {len(library_df)}')
    if final_threshold is not None:
        print(f'P{GOOD_PERCENTILE:.0f} threshold: {final_threshold:.3f}')
    print(f"Selected at P{GOOD_PERCENTILE:.0f}: {session_stats['accepted_this_session']}")

Generating candidate routes:   0%|          | 0/9945 [00:00<?, ?route/s]

Routes in library: 10000
P95 threshold: -3676.501
Selected at P95: 500


### 6.3 Finalize the top 95%
Run this cell after the batch finishes to flag the top 95% by the part 2 score and refresh the CSV/YAML snapshot.


In [ ]:
library_df = load_existing_library(OUTPUT_CSV)
if library_df.empty or 'screening_risk_adjusted_score' not in library_df.columns:
    raise RuntimeError('Run the batch cell first so the CSV log exists.')

score_values = library_df['screening_risk_adjusted_score'].to_numpy(dtype=float)
p95_threshold = float(np.percentile(score_values, GOOD_PERCENTILE))
library_df['selected_top95'] = library_df['screening_risk_adjusted_score'] >= p95_threshold
if 'validation_passes' in library_df.columns:
    library_df['selected_top95_and_validated'] = library_df['selected_top95'] & library_df['validation_passes'].fillna(False).astype(bool)
else:
    library_df['selected_top95_and_validated'] = library_df['selected_top95']

refresh_stats = {
    'accepted_this_session': int(library_df['selected_top95'].sum()),
    'screened_this_session': int(len(library_df)),
    'validated_this_session': int(library_df['validation_passes'].fillna(False).sum()) if 'validation_passes' in library_df.columns else 0,
    'duplicate_hits_this_session': int(library_df['is_duplicate'].fillna(False).sum()) if 'is_duplicate' in library_df.columns else 0,
    'attempts_this_session': int(len(library_df)),
}
library_df = write_library_snapshot(library_df, refresh_stats)
good_routes = library_df.loc[library_df['selected_top95']].copy().reset_index(drop=True)
print(f'Refreshed batch library from: {len(library_df):,} routes')
print(f'P{GOOD_PERCENTILE:.0f} threshold: {p95_threshold:.3f}')
print(f'Routes selected at P{GOOD_PERCENTILE:.0f}: {len(good_routes)}')
display(good_routes.head(10))


Refreshed batch library from: 40 routes
P95 threshold: -3374.175
Routes selected at P95: 2


,route_key,route_id,run_seed,route_seed,is_duplicate,screening_passes,selected_top95,route_node_ids,anchor_node_ids,ordered_anchor_node_ids,...,path_length_m,attempt_index,screening_average_gtc,screening_std_gtc,screening_risk_adjusted_score,screening_reward,screening_average_passenger_gtc_std,screening_n_tests,screening_coefficient_of_variation,selected_top95_and_validated
0,"[2154, 70, 2153, 3428, 890, 889, 891, 3629, 31...",B4L000001,1670006087,968396998,False,True,True,"[2154, 70, 2153, 3428, 890, 889, 891, 3629, 31...","[2154, 251, 7, 490]","[2154, 251, 7, 490]",...,66127.310604,2,1816.571275,0.0,-1816.571275,-1816.571275,6804.658329,1,0.0,True
1,"[131, 136, 134, 473, 880, 138, 69, 66, 68, 480...",B4L000019,1670006087,674005671,False,True,True,"[131, 136, 134, 473, 880, 138, 69, 66, 68, 480...","[131, 722, 1093, 101]","[131, 1093, 722, 101]",...,78407.244513,1,3134.815112,0.0,-3134.815112,-3134.815112,10495.017091,1,0.0,True


### 6.4 Inspect the current library
Use this cell to reload the persisted routes and inspect the top of the library after the batch finishes.


In [ ]:
current_library = load_existing_library(OUTPUT_CSV)
if current_library.empty:
    raise RuntimeError('Run the batch cell first so the CSV log exists.')

current_manifest = yaml.safe_load(OUTPUT_YAML.read_text(encoding='utf-8')) or {}
current_library = current_library.sort_values('screening_risk_adjusted_score', ascending=False, kind='stable').reset_index(drop=True)
print(f'Refreshed library from: {OUTPUT_CSV}')
print(f'Current library count: {len(current_library)}')
print(f'YAML manifest: {OUTPUT_YAML}')
print(current_manifest.get('summary', {}))
display(current_library.head(10))


Refreshed library from: C:\Users\lifei\OneDrive\Desktop\Thesis\Thesis Repository\Thesis\results\B4_good_routes\good_routes.csv
Current library count: 40
YAML manifest: C:\Users\lifei\OneDrive\Desktop\Thesis\Thesis Repository\Thesis\configs\B4_good_route_library.yaml
{'route_count': 40, 'mean_validation_average_gtc': None, 'mean_validation_std_gtc': None, 'mean_validation_risk_adjusted_score': None, 'std_validation_risk_adjusted_score': None, 'best_route_id': 'B4L000001', 'best_validation_risk_adjusted_score': None, 'best_validation_average_gtc': None, 'best_validation_std_gtc': None}


,route_key,route_id,run_seed,route_seed,is_duplicate,screening_passes,selected_top95,route_node_ids,anchor_node_ids,ordered_anchor_node_ids,...,path_length_m,attempt_index,screening_average_gtc,screening_std_gtc,screening_risk_adjusted_score,screening_reward,screening_average_passenger_gtc_std,screening_n_tests,screening_coefficient_of_variation,selected_top95_and_validated
0,"[2154, 70, 2153, 3428, 890, 889, 891, 3629, 31...",B4L000001,1670006087,968396998,False,True,True,"[2154, 70, 2153, 3428, 890, 889, 891, 3629, 31...","[2154, 251, 7, 490]","[2154, 251, 7, 490]",...,66127.310604,2,1816.571275,0.0,-1816.571275,-1816.571275,6804.658329,1,0.0,True
1,"[131, 136, 134, 473, 880, 138, 69, 66, 68, 480...",B4L000019,1670006087,674005671,False,True,True,"[131, 136, 134, 473, 880, 138, 69, 66, 68, 480...","[131, 722, 1093, 101]","[131, 1093, 722, 101]",...,78407.244513,1,3134.815112,0.0,-3134.815112,-3134.815112,10495.017091,1,0.0,True
2,"[89, 91, 366, 1108, 1107, 1110, 1112, 1118, 11...",B4L000006,1670006087,1040223205,False,True,False,"[89, 91, 366, 1108, 1107, 1110, 1112, 1118, 11...","[89, 1093, 87, 637]","[89, 1093, 87, 637]",...,32978.853256,1,3386.772974,0.0,-3386.772974,-3386.772974,11962.205364,1,0.0,False
3,"[3753, 3751, 3290, 3016, 3011, 882, 881, 883, ...",B4L000002,1670006087,429601073,False,False,False,"[3753, 3751, 3290, 3016, 3011, 882, 881, 883, ...","[3753, 2495, 3141, 1940]","[3753, 3141, 2495, 1940]",...,48647.962110,1,3762.724791,0.0,-3762.724791,-3762.724791,10702.937279,1,0.0,False
4,"[254, 253, 72, 74, 48, 49, 86, 87, 2101, 2097,...",B4L000018,1670006087,211982019,False,False,False,"[254, 253, 72, 74, 48, 49, 86, 87, 2101, 2097,...","[254, 24, 1768, 72]","[254, 24, 1768, 72]",...,16996.586165,1,4069.235389,0.0,-4069.235389,-4069.235389,13439.348805,1,0.0,False
5,"[91, 89, 92, 1150, 1149, 1144, 1142, 1145, 116...",B4L000012,1670006087,1800104183,False,False,False,"[91, 89, 92, 1150, 1149, 1144, 1142, 1145, 116...","[91, 619, 2083, 161]","[91, 2083, 619, 161]",...,34995.667276,1,4172.206501,0.0,-4172.206501,-4172.206501,12252.515594,1,0.0,False
6,"[1680, 1682, 1681, 1685, 3598, 2016, 60, 58, 6...",B4L000009,1670006087,630877302,False,False,False,"[1680, 1682, 1681, 1685, 3598, 2016, 60, 58, 6...","[1680, 92, 272, 342]","[1680, 92, 272, 342]",...,65223.336967,2,4227.182847,0.0,-4227.182847,-4227.182847,11956.265241,1,0.0,False
7,"[255, 256, 272, 1407, 1412, 1410, 1422, 1421, ...",B4L000018,1670006087,134325498,False,False,False,"[255, 256, 272, 1407, 1412, 1410, 1422, 1421, ...","[255, 1096, 1685, 873]","[255, 1096, 873, 1685]",...,23628.558121,1,4247.226703,0.0,-4247.226703,-4247.226703,12977.661457,1,0.0,False
8,"[476, 477, 486, 484, 485, 1278, 532, 490, 489,...",B4L000015,1670006087,618922183,False,False,False,"[476, 477, 486, 484, 485, 1278, 532, 490, 489,...","[476, 75, 2658, 686]","[476, 75, 2658, 686]",...,80009.101266,1,4373.820989,0.0,-4373.820989,-4373.820989,13543.150441,1,0.0,False
9,"[694, 711, 712, 708, 703, 702, 700, 680, 679, ...",B4L000008,1670006087,2083317522,False,False,False,"[694, 711, 712, 708, 703, 702, 700, 680, 679, ...","[694, 1047, 2506, 1249]","[694, 1047, 2506, 1249]",...,85448.660284,1,4460.445077,0.0,-4460.445077,-4460.445077,11815.913791,1,0.0,False


### 6.5 Geometric Analysis of Top Routes
Extract and visualize structural properties: compactness, path complexity, anchor spread.

In [19]:
import json
import math

# Load current library
current_library = load_existing_library(OUTPUT_CSV)
if current_library.empty:
    print("No routes in library yet. Run part 3 batch generation first.")
else:
    top_routes = current_library.loc[current_library.get('selected_top95', False) == True].copy()
    if top_routes.empty:
        top_routes = current_library.head(10)  # Fallback to top 10
    
    def parse_latlons(json_str):
        """Parse JSON-encoded lat/lon coordinates."""
        if isinstance(json_str, str):
            return json.loads(json_str)
        return json_str
    
    def compute_anchor_spread(anchor_latlon):
        """Compute max distance between any two anchors (meters)."""
        if len(anchor_latlon) < 2:
            return 0.0
        coords = [parse_latlons(x) if isinstance(x, str) else x for x in anchor_latlon]
        max_dist = 0.0
        for i in range(len(coords)):
            for j in range(i + 1, len(coords)):
                lat1, lon1 = coords[i]
                lat2, lon2 = coords[j]
                dlat = (lat2 - lat1) * 111000
                dlon = (lon2 - lon1) * 111000 * math.cos(math.radians(lat1))
                dist = math.sqrt(dlat**2 + dlon**2)
                max_dist = max(max_dist, dist)
        return max_dist
    
    def compute_sinuosity(path_length, anchor_latlon):
        """Estimate sinuosity from path length vs anchor span."""
        spread = compute_anchor_spread(anchor_latlon)
        if spread > 0:
            return path_length / spread
        return 1.0
    
    # Extract geometry metrics
    geometry_rows = []
    for _, row in top_routes.iterrows():
        anchor_latlon = parse_latlons(row['anchor_latlon']) if isinstance(row['anchor_latlon'], str) else row['anchor_latlon']
        path_length = float(row['path_length_m'])
        area = float(row['polygon_area_m2'])
        
        compactness = area / max(path_length, 1.0)
        anchor_spread = compute_anchor_spread(anchor_latlon)
        sinuosity = compute_sinuosity(path_length, anchor_latlon)
        nodes = row.get('route_node_ids')
        if isinstance(nodes, str):
            nodes = json.loads(nodes)
        node_count = len(nodes) if nodes else 0
        
        geometry_rows.append({
            'route_id': row['route_id'],
            'area_m2': area,
            'path_length_m': path_length,
            'compactness': compactness,
            'anchor_spread_m': anchor_spread,
            'sinuosity': sinuosity,
            'node_count': node_count,
            'screening_score': float(row['screening_risk_adjusted_score']),
        })
    
    geometry_df = pd.DataFrame(geometry_rows)
    
    print(f"Top {len(top_routes)} routes geometry summary:")
    print(f"  Area: {geometry_df['area_m2'].mean():.0f} ± {geometry_df['area_m2'].std():.0f} m²")
    print(f"  Length: {geometry_df['path_length_m'].mean():.0f} ± {geometry_df['path_length_m'].std():.0f} m")
    print(f"  Compactness: {geometry_df['compactness'].mean():.1f} ± {geometry_df['compactness'].std():.1f}")
    print(f"  Anchor spread: {geometry_df['anchor_spread_m'].mean():.0f} ± {geometry_df['anchor_spread_m'].std():.0f} m")
    print(f"  Sinuosity: {geometry_df['sinuosity'].mean():.2f} ± {geometry_df['sinuosity'].std():.2f}")
    print(f"  Nodes: {geometry_df['node_count'].mean():.0f} ± {geometry_df['node_count'].std():.0f}")
    
    display(geometry_df)

Top 10 routes geometry summary:
  Area: 56923013 ± 60919598 m²
  Length: 55647 ± 32644 m
  Compactness: 743.7 ± 603.1
  Anchor spread: 20604 ± 12061 m
  Sinuosity: 2.67 ± 0.17
  Nodes: 248 ± 108


,route_id,area_m2,path_length_m,compactness,anchor_spread_m,sinuosity,node_count,screening_score
0,ROUTE000001,2.230534e+06,17099.572489,130.443827,6768.577224,2.526317,180,-7320.503363
1,ROUTE000002,3.969208e+06,14359.110906,276.424353,5511.196844,2.605443,141,-6358.773329
2,ROUTE000003,3.675454e+07,65504.626449,561.098343,24240.053899,2.702330,304,-5978.892943
3,ROUTE000004,1.464816e+08,83839.053993,1747.175703,27811.409276,3.014556,295,-8215.426363
4,ROUTE000005,1.398209e+07,48701.273751,287.099121,18374.111976,2.650538,135,-7287.572907
5,ROUTE000006,1.745267e+07,48124.879354,362.653895,16665.798277,2.887643,289,-4649.321673
6,ROUTE000007,1.572698e+08,97746.250535,1608.959777,37086.573168,2.635624,366,-5356.159519
7,ROUTE000008,9.372141e+07,81159.931580,1154.774418,32510.077569,2.496455,240,-5454.712861
8,ROUTE000009,2.767288e+06,11784.981230,234.814812,4708.728017,2.502795,97,-6199.293369
9,ROUTE000010,9.460104e+07,88149.966399,1073.182973,32365.128612,2.723609,430,-3961.810785


### 6.6 Comparison with Part 2 (Rejection Sampling)
Compare batch results against the 5000-sample rejection sampling baseline.

In [20]:
# Load Part 2 results for comparison
try:
    part2_manifest = yaml.safe_load(PART2_CONFIG.read_text(encoding='utf-8')) or {}
    part2_summary = part2_manifest.get('summary', {})
    
    # Get top routes from current library
    if 'geometry_df' not in dir():
        print("Run 6.5 first to generate geometry_df")
    else:
        comparison_data = {
            'metric': [
                'Sample size',
                'Best score',
                'Mean score (top 95%)',
                'Std score (top 95%)',
                'Good routes count',
                'Elite routes count (P99)',
            ],
            'Part 2 (P95)': [
                f"{part2_manifest.get('sample_size', 'N/A'):,}",
                f"{part2_summary.get('best_risk_adjusted_score', 'N/A'):.3f}" if isinstance(part2_summary.get('best_risk_adjusted_score'), (int, float)) else "N/A",
                f"{part2_summary.get('validation_mean_risk_adjusted_score', 'N/A'):.3f}" if isinstance(part2_summary.get('validation_mean_risk_adjusted_score'), (int, float)) else "N/A",
                f"{part2_summary.get('validation_std_risk_adjusted_score', 'N/A'):.3f}" if isinstance(part2_summary.get('validation_std_risk_adjusted_score'), (int, float)) else "N/A",
                f"{part2_summary.get('good_route_count', 'N/A')}",
                f"{part2_summary.get('elite_route_count', 'N/A')}",
            ],
            'This Batch': [
                f"{len(current_library) if not current_library.empty else 0}",
                f"{current_library['screening_risk_adjusted_score'].max():.3f}" if not current_library.empty and 'screening_risk_adjusted_score' in current_library.columns else "N/A",
                f"{geometry_df['screening_score'].mean():.3f}" if len(top_routes) > 0 else "N/A",
                f"{geometry_df['screening_score'].std():.3f}" if len(top_routes) > 1 else "N/A",
                f"{len(top_routes) if not current_library.empty else 0}",
                f"{len(top_routes) if not current_library.empty else 0}",
            ],
        }
        
        comp_df = pd.DataFrame(comparison_data)
        print("Part 2 vs This Batch:")
        display(comp_df)
except Exception as e:
    print(f"Could not load Part 2 config: {e}")

Part 2 vs This Batch:


,metric,Part 2 (P95),This Batch
0,Sample size,"5,000",10000
1,Best score,-0.000,-168.683
2,Mean score (top 95%),-7041.345,-6078.247
3,Std score (top 95%),605.295,1295.623
4,Good routes count,249,10
5,Elite routes count (P99),49,10


### 6.7 Route Diversity Analysis
Filter near-duplicate routes by anchor combinations to avoid redundant storage.

In [21]:
# Detect duplicate anchor combinations in top routes
if not current_library.empty:
    if 'selected_top95' in current_library.columns:
        top_routes_check = current_library.loc[current_library['selected_top95'] == True].copy()
    else:
        top_routes_check = current_library.head(10)
    
    if len(top_routes_check) > 0:
        anchor_sigs = {}
        duplicates = []
        
        for _, row in top_routes_check.iterrows():
            anchor_ids = row['anchor_node_ids']
            if isinstance(anchor_ids, str):
                anchor_ids = json.loads(anchor_ids)
            sig = tuple(sorted(int(x) for x in anchor_ids))
            
            if sig not in anchor_sigs:
                anchor_sigs[sig] = []
            anchor_sigs[sig].append({
                'route_id': row['route_id'],
                'score': row['screening_risk_adjusted_score'],
            })
        
        # Find anchor combinations with >1 route
        for sig, routes in anchor_sigs.items():
            if len(routes) > 1:
                duplicates.append({
                    'anchor_combo': sig,
                    'count': len(routes),
                    'best_route': routes[0]['route_id'],
                    'best_score': routes[0]['score'],
                    'all_routes': [r['route_id'] for r in routes],
                })
        
        if duplicates:
            dup_df = pd.DataFrame(duplicates)
            print(f"Found {len(duplicates)} anchor combinations with >1 route in top 95%:")
            display(dup_df)
            print("\nNote: Keep best route per anchor combo, optionally merge others into validation queue.")
        else:
            print(f"All {len(top_routes_check)} top routes have unique anchor combinations")
    else:
        print("No top routes to analyze for diversity.")
else:
    print("No routes in library. Run batch generation first.")

No top routes to analyze for diversity.


### 6.8 Export Top Routes & Summary
Save top routes with metadata for later validation, clustering, or visualization.

In [22]:
# Export top routes with geometry metadata
if not current_library.empty:
    if 'selected_top95' in current_library.columns:
        export_routes = current_library.loc[current_library['selected_top95'] == True].copy()
    else:
        export_routes = current_library.head(10)
    
    if len(export_routes) > 0 and 'geometry_df' in dir():
        export_rows = []
        for idx, (_, row) in enumerate(export_routes.iterrows()):
            geom = geometry_df.iloc[idx] if idx < len(geometry_df) else None
            export_rows.append({
                'rank': idx + 1,
                'route_id': row['route_id'],
                'screening_score': row['screening_risk_adjusted_score'],
                'area_m2': geom['area_m2'] if geom is not None else row['polygon_area_m2'],
                'length_m': geom['path_length_m'] if geom is not None else row['path_length_m'],
                'compactness': geom['compactness'] if geom is not None else None,
                'anchor_spread_m': geom['anchor_spread_m'] if geom is not None else None,
                'sinuosity': geom['sinuosity'] if geom is not None else None,
                'node_count': geom['node_count'] if geom is not None else None,
                'route_seed': row['route_seed'],
                'attempt_index': row['attempt_index'],
                'anchor_ids': str(row['anchor_node_ids']),
            })
        
        export_df = pd.DataFrame(export_rows)
        
        # Save as CSV summary
        export_path = OUTPUT_DIR / 'top_routes_summary.csv'
        export_df.to_csv(export_path, index=False)
        print(f"Exported {len(export_df)} top routes to: {export_path}")
        
        # Save as JSONL for detailed access (preserve lists/arrays)
        jsonl_path = OUTPUT_DIR / 'top_routes_detailed.jsonl'
        with open(jsonl_path, 'w', encoding='utf-8') as f:
            for _, row in export_routes.iterrows():
                detail = {k: (json.loads(v) if isinstance(v, str) and k in ['route_node_ids', 'anchor_node_ids', 'path_latlon', 'anchor_latlon'] else v) for k, v in row.to_dict().items()}
                f.write(json.dumps(detail, ensure_ascii=False, default=str) + '\n')
        print(f"Exported detailed records to: {jsonl_path}")
        
        # Summary stats
        print(f"\nTop {len(export_df)} Routes Summary:")
        print(f"  Score range: {export_df['screening_score'].min():.3f} to {export_df['screening_score'].max():.3f}")
        print(f"  Median area: {export_df['area_m2'].median():.0f} m²")
        print(f"  Median length: {export_df['length_m'].median():.0f} m")
        if 'sinuosity' in export_df.columns and export_df['sinuosity'].notna().any():
            print(f"  Median sinuosity: {export_df['sinuosity'].median():.2f}")
        
        display(export_df)
    else:
        print("Run 6.5 first to generate geometry_df")
else:
    print("No routes in library. Run batch generation first.")

Run 6.5 first to generate geometry_df
